# Chapter 21 — Distance and Margin: kNN and SVMs

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Data

This chapter reuses `customers.csv`, created in Chapter 24. Run that notebook first, or just run the generator below — it is the same code.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(24)
n = 8000

city = rng.choice([f"CITY_{i:03d}" for i in range(180)], n)      # high cardinality
plan = rng.choice(["basic", "plus", "pro"], n, p=[.55, .32, .13])
channel = rng.choice(["web", "app", "phone"], n, p=[.5, .38, .12])
signup = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    rng.integers(0, 730, n), unit="D")
income = np.round(np.exp(rng.normal(10.2, 0.55, n)), 0)
sessions = rng.poisson(6, n)
basket = np.round(np.exp(rng.normal(3.1, 0.7, n)), 2)

# churn depends on plan, engagement, and value-for-money -- not on city
z = (-0.4
     - 0.55 * (plan == "pro") + 0.35 * (plan == "basic")
     - 0.09 * sessions
     + 1.85 * (basket / (income / 1000) > 1.4)
     + 0.30 * (channel == "phone")
     + rng.normal(0, 0.6, n))
churn = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

df = pd.DataFrame({"City": city, "Plan": plan, "Channel": channel,
                   "SignupDate": signup.strftime("%Y-%m-%d"),
                   "AnnualIncome": income, "Sessions": sessions,
                   "AvgBasket": basket, "Churn": churn})
df.loc[rng.random(n) < 0.09, "AnnualIncome"] = np.nan     # real gaps
df.to_csv("customers.csv", index=False)
print(f"wrote customers.csv: {n:,} customers, churn {churn.mean():.1%}, "
      f"{df.City.nunique()} cities, {df.AnnualIncome.isna().sum()} missing incomes")

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, pandas as pd, warnings, time; warnings.filterwarnings("ignore")
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.datasets import make_moons, make_circles
df = pd.read_csv("customers.csv", parse_dates=["SignupDate"])
y = df.pop("Churn").values
df = df.drop(columns=["City", "SignupDate"])
NUM = ["AnnualIncome", "Sessions", "AvgBasket"]
CAT = ["Plan", "Channel"]
def pipe(clf, scale=True):
    steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("sc", StandardScaler()))
    return Pipeline([("pre", ColumnTransformer([
                        ("n", Pipeline(steps), NUM),
                        ("c", OneHotEncoder(handle_unknown="ignore"), CAT)])),
                     ("clf", clf)])
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
# kNN has no training step. It memorizes, then measures distance at
# prediction time -- which makes scaling decisive, not merely advisable.
for label, scale in [("unscaled", False), ("standardized", True)]:
    s = cross_val_score(pipe(KNeighborsClassifier(25), scale=scale),
                        df, y, cv=cv, scoring="roc_auc")
    print(f"kNN, {label:<13} CV AUC {s.mean():.4f} +/- {s.std():.4f}")

print(f"\nwhy: the numeric columns before scaling")
for c in NUM:
    col = df[c].dropna()
    print(f"  {c:<16} sd {col.std():>10,.1f}")

### Block 2  (`c2.py`)

In [ ]:
# k trades bias against variance, exactly as Chapter 17 described.
print(f"{'k':>5}{'train AUC':>12}{'CV AUC':>10}{'sd':>8}")
Xtr, Xte, ytr, yte = train_test_split(df, y, test_size=0.3,
                                      random_state=0, stratify=y)
for k in (1, 5, 15, 35, 75, 200):
    m = pipe(KNeighborsClassifier(k)).fit(Xtr, ytr)
    tr = roc_auc_score(ytr, m.predict_proba(Xtr)[:, 1])
    s = cross_val_score(pipe(KNeighborsClassifier(k)), Xtr, ytr,
                        cv=cv, scoring="roc_auc")
    print(f"{k:>5}{tr:>12.4f}{s.mean():>10.4f}{s.std():>8.4f}")

### Block 3  (`c3.py`)

In [ ]:
# Distances stop discriminating as dimensions grow. This is the reason
# kNN degrades on wide data, and it is arithmetic rather than folklore.
rng = np.random.default_rng(0)
print(f"{'dimensions':>11}{'nearest':>10}{'farthest':>10}"
      f"{'ratio':>9}{'contrast':>11}")
for d in (2, 5, 20, 100, 500):
    X = rng.uniform(size=(2000, d))
    q = rng.uniform(size=(1, d))
    dist = np.sqrt(((X - q) ** 2).sum(1))
    near, far = dist.min(), dist.max()
    print(f"{d:>11}{near:>10.3f}{far:>10.3f}{far/near:>9.2f}"
          f"{(far-near)/near:>11.3f}")
print("\nwhen the farthest point is barely further than the nearest,")
print("'nearest neighbour' has stopped meaning anything.")

### Block 4  (`c4.py`)

In [ ]:
# An SVM maximizes the margin: the gap between the boundary and the
# closest points on either side. Only those points matter.
Xm, ym = make_moons(n_samples=800, noise=0.18, random_state=0)
Xc, yc = make_circles(n_samples=800, noise=0.10, factor=0.45, random_state=0)

for name, X_, y_ in [("moons", Xm, ym), ("circles", Xc, yc)]:
    print(f"{name}")
    for label, clf in [("logistic", LogisticRegression(max_iter=2000)),
                       ("linear SVM", LinearSVC(C=1.0, max_iter=20000)),
                       ("SVM, RBF kernel", SVC(C=1.0, kernel="rbf"))]:
        m = make_pipeline(StandardScaler(), clf)
        s = cross_val_score(m, X_, y_, cv=cv, scoring="accuracy")
        extra = ""
        if isinstance(clf, SVC):
            m.fit(X_, y_)
            extra = f"   support vectors {m[-1].n_support_.sum()} of {len(y_)}"
        print(f"  {label:<18}accuracy {s.mean():.4f}{extra}")

### Block 5  (`c5.py`)

In [ ]:
# C and gamma both control complexity, and they interact. C sets how much
# margin violation is tolerated; gamma sets how far one point's influence
# reaches. Large values of either overfit.
Xm, ym = make_moons(n_samples=600, noise=0.28, random_state=0)
print(f"{'gamma':>8}" + "".join(f"{'C='+str(c):>10}" for c in
                                (0.1, 1, 10, 100)))
for g in (0.1, 1.0, 10.0, 100.0):
    row = ""
    for C in (0.1, 1, 10, 100):
        s = cross_val_score(make_pipeline(StandardScaler(),
                            SVC(C=C, gamma=g)), Xm, ym, cv=cv,
                            scoring="accuracy").mean()
        row += f"{s:>10.4f}"
    print(f"{g:>8.1f}{row}")
print("\nbest cell is the one to use; the corners show both failure modes.")

### Block 6  (`c6.py`)

In [ ]:
# What all this costs. kNN has no training time and expensive prediction;
# a kernel SVM has expensive training and scales badly with rows.
rng = np.random.default_rng(0)
print(f"{'rows':>8}{'kNN fit':>10}{'kNN predict':>13}"
      f"{'SVM fit':>10}{'SVM predict':>13}")
for n in (1000, 4000, 16000):
    X = rng.normal(size=(n, 12))
    yy = (X[:, 0] + X[:, 1] ** 2 > 1).astype(int)
    Xq = rng.normal(size=(500, 12))
    row = []
    for clf in (KNeighborsClassifier(25), SVC(kernel="rbf")):
        t0 = time.perf_counter(); clf.fit(X, yy)
        t_fit = time.perf_counter() - t0
        t0 = time.perf_counter(); clf.predict(Xq)
        t_pred = time.perf_counter() - t0
        row += [t_fit, t_pred]
    print(f"{n:>8}{row[0]*1000:>9.1f}ms{row[1]*1000:>12.1f}ms"
          f"{row[2]*1000:>9.1f}ms{row[3]*1000:>12.1f}ms")